# IS 4487 Assignment 6: Data Cleaning with Airbnb Listings

In this assignment, you will:
- Load a raw Airbnb listings dataset
- Identify and resolve missing or inconsistent data
- Decide what data to drop, keep, or clean
- Save a clean dataset to use in Assignment 7

## Why This Matters

Data cleaning is one of the most important steps in any analysis — but it's often the least visible. Airbnb hosts, managers, and policy teams rely on clean data to make decisions. This assignment gives you experience cleaning raw data and justifying your choices so others can understand your process.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_06_data_cleaning.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>



## Dataset Description

The dataset you'll be using is a **detailed Airbnb listing file**, available from [Inside Airbnb](https://insideairbnb.com/get-the-data/).

Each row represents one property listing. The columns include:

- **Host attributes** (e.g., host ID, host name, host response time)
- **Listing details** (e.g., price, room type, minimum nights, availability)
- **Location data** (e.g., neighborhood, latitude/longitude)
- **Property characteristics** (e.g., number of bedrooms, amenities, accommodates)
- **Calendar/booking variables** (e.g., last review date, number of reviews)

📌 The schema is consistent across cities, so you can expect similar columns regardless of the location you choose.


## 1. Choose a City & Upload Your Dataset

📥 Follow these steps:

1. Go to: [https://insideairbnb.com/get-the-data/](https://insideairbnb.com/get-the-data/)
2. Choose a city you’re interested in.
3. Download the file named: **`listings.csv.gz`** under that city.
4. In your notebook:
   - Open the left sidebar
   - Click the folder icon 📁
   - Click the upload icon ⬆️ and choose your `listings.csv.gz` file
5. Use the file path `/content/listings.csv.gz` when loading your data.
6. Import standard libraries (`pandas`, `numpy`, `seaborn`, `matplotlib`)


In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [5]:
df = pd.read_csv('/content/listings.csv.gz')

print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (92638, 90)


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,11551,https://www.airbnb.com/rooms/11551,20260619061251,2026-06-25,city scrape,Artistic London Pied-à-Terre,Live like a local in this quintessential Londo...,NaN,https://a0.muscache.com/pictures/prohost-api/H...,43039,...,4.84,4.54,4.49,NaN,NaN,1,1,0,0,0.99
1,13913,https://www.airbnb.com/rooms/13913,20260619061251,2026-06-25,city scrape,Holiday London DB Room Let-on going,My bright double bedroom with a large window h...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,54730,...,4.88,4.79,4.79,NaN,NaN,2,1,1,0,0.30
2,15400,https://www.airbnb.com/rooms/15400,20260619061251,2026-06-26,city scrape,Bright Chelsea Apartment. Chelsea!,Lots of windows and light. St Luke's Gardens ...,NaN,https://a0.muscache.com/pictures/428392/462d26...,60302,...,4.84,4.94,4.74,NaN,NaN,1,1,0,0,0.49
3,17402,https://www.airbnb.com/rooms/17402,20260619061251,2026-06-29,city scrape,Spacious Fitzrovia 3bed/2bath By Soho & Oxford St,"Stay in this spacious 3-bedroom, 2-bath apartm...",NaN,https://a0.muscache.com/pictures/39d5309d-fba7...,67564,...,4.73,4.89,4.62,NaN,NaN,2,2,0,0,0.31
4,513198,https://www.airbnb.com/rooms/513198,20260619061251,2026-06-29,city scrape,Stunning views over Central London! (room),Wake up to the sound of Big Ben!<br /><br />Th...,NaN,https://a0.muscache.com/pictures/6442310/e4b3e...,2528030,...,4.74,4.72,4.42,NaN,NaN,2,1,1,0,0.25


## 2. Explore Missing Values

### Business framing:

Stakeholders don’t like surprises in the data. Missing values can break dashboards, confuse pricing models, or create blind spots for host managers.

### Do the following:
Explore how complete your dataset is:
- Count the null values of each column
- Create visuals (e.g. heatmaps, boxplots, bar charts, etc) to help show what columns are missing values
- Keep in mind which column(s) are missing too much data, you will delete these in the next step

In [12]:
missing_counts = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_summary = (
    pd.DataFrame(
        {"Missing Count": missing_counts, "Missing %": missing_percentage}
    )
    .sort_values(by="Missing Count", ascending=False)
    .filter(
        items=missing_counts[missing_counts > 0].index, axis=0
    )
)
print(missing_summary)

                              Missing Count   Missing %
description                            2089    2.255014
neighborhood_overview                 92638  100.000000
picture_url                               5    0.005397
host_profile_id                          76    0.082040
host_profile_url                         84    0.090676
host_name                                76    0.082040
host_since                            92638  100.000000
hosts_time_as_user_years                 76    0.082040
hosts_time_as_user_months                76    0.082040
hosts_time_as_host_years                 77    0.083119
hosts_time_as_host_months                77    0.083119
host_location                         22710   24.514778
host_about                            43409   46.858740
host_response_time                    92638  100.000000
host_response_rate                    92638  100.000000
host_acceptance_rate                  92638  100.000000
host_is_superhost                        76    0

### Reflection:
1. What are the top 3 columns with the most missing values?
2. Which ones are likely to create business issues?
3. Which could be safely ignored or dropped?

### ✍️ Your Response: 🔧
1. The colums with the most missing values are insant_bookable, license,calendar_updated, neighborhood, neighborhood_group_cleansed, host_verification, host_total_listing_count, host_neighborhood, host_thumbnail_url, host_response_time,
host_response_rate,
host_acceptance_rate, host_since, and neighborhood_overview. These all have 100% of their values missing.
2. The ones most likely to create business issues are columns like price that has around 30% of its data missing, the review_score_rating that has about 24% missing as having these missing could hurt pricing and recommendations across the platform.

3. The columns that could be safely dropped and ignored are columns that have no data like calendar_updated or license since they do not have to do with customer operations and do not hurt Airbnb terribly if they are gone.


## 3. Drop Columns That Aren’t Useful

### Business framing:  

Not every column adds value. Analysts often remove columns that are too empty, irrelevant, or repetitive — especially when preparing data for others.

### Do the following:
Make a decision:

- Choose 2–4 columns to drop from your dataset
- Document your reasons for each one
- Confirm they're gone with `.head()` or `.info()`

In [13]:
columns_to_drop = ['calendar_updated', 'license', 'neighborhood_group_cleansed', 'neighborhood_overview']

df.drop(columns=columns_to_drop, inplace=True, errors='ignore')

print(f"New dataset shape: {df.shape}")
print("Are the dropped columns still in the dataframe?")
for col in columns_to_drop:
    print(f" - {col}: {'Yes' if col in df.columns else 'No'}")
df.head()

New dataset shape: (92638, 87)
Are the dropped columns still in the dataframe?
 - calendar_updated: No
 - license: No
 - neighborhood_group_cleansed: No
 - neighborhood_overview: No


,id,listing_url,scrape_id,last_scraped,source,name,description,picture_url,host_id,host_url,...,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,11551,https://www.airbnb.com/rooms/11551,20260619061251,2026-06-25,city scrape,Artistic London Pied-à-Terre,Live like a local in this quintessential Londo...,https://a0.muscache.com/pictures/prohost-api/H...,43039,https://www.airbnb.com/users/show/43039,...,4.77,4.84,4.54,4.49,NaN,1,1,0,0,0.99
1,13913,https://www.airbnb.com/rooms/13913,20260619061251,2026-06-25,city scrape,Holiday London DB Room Let-on going,My bright double bedroom with a large window h...,https://a0.muscache.com/pictures/miso/Hosting-...,54730,https://www.airbnb.com/users/show/54730,...,4.82,4.88,4.79,4.79,NaN,2,1,1,0,0.30
2,15400,https://www.airbnb.com/rooms/15400,20260619061251,2026-06-26,city scrape,Bright Chelsea Apartment. Chelsea!,Lots of windows and light. St Luke's Gardens ...,https://a0.muscache.com/pictures/428392/462d26...,60302,https://www.airbnb.com/users/show/60302,...,4.88,4.84,4.94,4.74,NaN,1,1,0,0,0.49
3,17402,https://www.airbnb.com/rooms/17402,20260619061251,2026-06-29,city scrape,Spacious Fitzrovia 3bed/2bath By Soho & Oxford St,"Stay in this spacious 3-bedroom, 2-bath apartm...",https://a0.muscache.com/pictures/39d5309d-fba7...,67564,https://www.airbnb.com/users/show/67564,...,4.73,4.73,4.89,4.62,NaN,2,2,0,0,0.31
4,513198,https://www.airbnb.com/rooms/513198,20260619061251,2026-06-29,city scrape,Stunning views over Central London! (room),Wake up to the sound of Big Ben!<br /><br />Th...,https://a0.muscache.com/pictures/6442310/e4b3e...,2528030,https://www.airbnb.com/users/show/2528030,...,4.79,4.74,4.72,4.42,NaN,2,1,1,0,0.25


### Reflection:
1. Which columns did you drop?
2. Why were they not useful from a business perspective?
3. What could go wrong if you left them in?

### ✍️ Your Response: 🔧
1. The colums that I dropped were calendar_updated,license, neighborhood_group_cleansed, and neighborhood_overview.

2. These columns are not useful from a business perspective as Airbnb analytically does not need to know the details of the neighborhood and if it is grouped or certain attributes about their hosts to successfully run their business.

3. If we left these columns in and they were attached with other data, it could severely skew or mess up other analytics that need to be done.



## 4. Fill or Fix Values in Key Columns

### Business framing:  

Let’s say your manager wants to see a map of listings with prices and review scores. If key fields are blank, the map won’t work. But not all missing values should be filled the same way.

### Do the following:
- Choose 2 columns with missing values
- Use a strategy to fill or flag those values
  - (e.g., median, “unknown”, forward-fill, or a placeholder)
- Explain what you did and why

In [14]:

df['description'] = df['description'].fillna("No description provided")

median_rating = df['review_scores_rating'].median()
df['review_scores_rating'] = df['review_scores_rating'].fillna(median_rating)

print(f"Remaining missing values in 'description': {df['description'].isnull().sum()}")
print(f"Remaining missing values in 'review_scores_rating': {df['review_scores_rating'].isnull().sum()}")

Remaining missing values in 'description': 0
Remaining missing values in 'review_scores_rating': 0


### Reflection:
1. What two columns did you clean?
2. What method did you use for each, and why?
3. What risks are there in how you filled the data?

### ✍️ Your Response: 🔧
1. The two columns I cleaned were the description and the review_scores_rating.

2. For the description, I replaced the missing values with a placeholder that said no description provided so it didnt mess other data up, and for the review_scores_rating, I used the median of the column to fill in the missing ratings so it doesn't create any extreme outliers and makes the distribution clean.

3. The risks that are associated with how I filled in the data are that it can create a high frequency category of no description provided variables and imputing the mean to fill the missing ratings can distort relationships, distort the distributions, artifically reduce the variance, and give a property a false and/or over-estimated rating.


## 5. Convert and Clean Data Types

### Business framing:  

Sometimes columns that look like numbers are actually stored as text — which breaks calculations and slows down analysis. Common examples are price columns with dollar signs or availability stored as strings.

### Do the following:
- Identify one column with the wrong data type
- Clean and convert it into a usable format (e.g., from string to number)
- Check your work by summarizing or plotting the cleaned column

In [15]:
df['price'] = df['price'].astype(str).str.replace('$', '').str.replace(',', '').astype(float)

print("Price data type:", df['price'].dtype)
print(df['price'].describe())

Price data type: float64
count     62240.000000
mean        271.469220
std        2190.599006
min           2.230000
25%         100.000000
50%         180.000000
75%         300.032500
max      527524.000000
Name: price, dtype: float64


### Reflection: :
1. What column did you fix?
2. What cleaning steps did you apply?
3. How does this help prepare the data for later use?

### ✍️ Your Response: 🔧
1. The column i chose to fix was Price

2.The cleaning steps that I applied to this column was to remove the $ symbol and the comma so it is converted into a numerical data value.

3. This helps prepare the data for later use so when we are doing analysis and making charts, etc. the values are consistent and uniform.

## 6. Remove Duplicate Records

### Business framing:  

If a listing appears twice, it could inflate revenue estimates or confuse users. Airbnb needs each listing to be unique and accurate.

### Do the following:
- Check for rows that are exact duplicates
- If your data has an ID column and each ID is supposed to unique, then make sure there are no duplicate IDs
- Remove duplicates if found

In [18]:
exact_duplicates = df.duplicated().sum()
print(f"Number of exact duplicate rows: {exact_duplicates}")

duplicate_ids = df.duplicated(subset=['id']).sum()
print(f"Number of duplicate listing IDs: {duplicate_ids}")

if exact_duplicates > 0 or duplicate_ids > 0:
    df.drop_duplicates(subset=['id'], keep='first', inplace=True)
    print()
    print(f"New dataset shape: {df.shape}")
else:
    print()

Number of exact duplicate rows: 0
Number of duplicate listing IDs: 0



### Reflection:
1. Did you find duplicates?
2. How did you decide what to drop or keep?
3. Why are duplicates risky for Airbnb teams?

### ✍️ Your Response: 🔧 🔧
1. I did not find any duplicate data in the dataset I grabbed.

2. I would have decided what to keep and what to keep and what to get rid of based on what Airbnb would/could have used to make their processes more efficient and help the customers on their platform. Data that doesn't fall into this bin could have been deleted.

3. Duplicates are risky for Airbnb as if there is a duplicate listing, for example, then customers could get confused or be under the impression that it is a false posting. This could skew their metrics or accidently double book a property.

## 7. Export Cleaned Data

Before wrapping up, export your cleaned Airbnb dataset to a CSV file. You'll need this file for **Assignment 7**, where you'll perform data transformation techniques.

### Do the following:
Make sure your data has:
- Cleaned and consistent column values
- Proper data types for each column
- Any unnecessary columns removed

This file should be the version of your dataset that you’d feel confident sharing with a teammate or using for deeper analysis.



```
# Explanation:
# - "cleaned_airbnb_data_6.csv" is the name of the file that will be saved
# - index=False prevents pandas from writing row numbers into the CSV
# - The file will be saved to your working directory (in Colab, you'll need to download it manually. Once you see the data in your files tab, just click on the three dots, then click “download”)
# - YOU MAY NEED TO PRESS “RUN” MULTIPLE TIMES IN ORDER FOR IT TO SHOW UP
# - FOR SOME DEVICES, IT MAY TAKE A FEW MINUTES BEFORE YOUR FILE SHOWS UP

```





In [21]:
print("--- Dataset Info & Data Types ---")
df.info()

print("\n--- Missing Values Check ---")
print(df.isnull().sum()[df.isnull().sum() > 0])
output_filename = "cleaned_airbnb_data_6.csv"
df.to_csv(output_filename, index=False)

print(f"\nSuccessfully exported cleaned dataset to '{output_filename}'!")

--- Dataset Info & Data Types ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92638 entries, 0 to 92637
Data columns (total 87 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            92638 non-null  int64  
 1   listing_url                                   92638 non-null  object 
 2   scrape_id                                     92638 non-null  int64  
 3   last_scraped                                  92638 non-null  object 
 4   source                                        92638 non-null  object 
 5   name                                          92638 non-null  object 
 6   description                                   92638 non-null  object 
 7   picture_url                                   92633 non-null  object 
 8   host_id                                       92638 non-null  int64  
 9   host_url                   

## 8. Final Reflection

You’ve just cleaned a real-world Airbnb dataset — the kind of work that happens every day in analyst and data science roles.

Before you move on to data transformation in Assignment 7, take a few moments to reflect on the decisions you made and what you learned.

### Reflection:
1. Which section of the assignment was the most challenging for you? What made it challenging? Was it the coding or the application of class concepts?
2. Look back at your assignment and discuss a time when you had to drop, fix, or keep a piece of data. Why was it necessary to drop, fix, or keep that specific piece of data? (e.x. dropping a column full of null values because it has no useful information)
3. What’s one way a business team (e.g., hosts, pricing analysts, app administrators) might benefit from the cleaned version of this data?
4. If you had more time, what would you explore or clean further?
5. How does this relate to your customized learning outcome you created in canvas?


Write your response clearly in full sentences. No more than a few sentences required per response.


### ✍️ Your Response: 🔧

1. The section of the assignment that was the most challenging for me was converting and cleaning the data. It was challenging as sometimes I did not know what data I exactly wanted to clean or how I wanted to clean it. I think it was a mixture of the coding and the class concepts becuase I had to think about how cleaning each variable would look for the rest of the data.
2. During Lab 6, there was a lot of data fields with missing data or no data at all and it ended up better for the data to remove them entirely then trying to fill them in with something that would not have been helpful.
3. One way that a business team might utilize this cleaned data is so their processes can be more accurate and reliable. When decision making, if they get an inflated or deflated number or insight becuase of messy data, a business could accidently do something that would harm their busines
4. I would have cleaned more data and gotten rid of any more entirely null columns and filled some appropriate columns to make the data more useful.
5. This relates to the customized learning outcome that I have in Canvas as the exact wording of the first one is "I can take data from any step of the process and remove errors and inconsistencies


## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [ ]:
!jupyter nbconvert --to html "assignment_06_data_cleaning.ipynb"